# Notebook of new fokl_to_pyomo function supporting ODE

1. convert GP to dummy Pyomo model (with only normalized 'x' inputs variables automatically generated)
1. merge dummy Pyomo model with global model
1. apply normalization to user-created input variables; or, if not user-created, use ```pyo.Var()``` by default

---

In [1]:
from FoKL import FoKLRoutines
import pyomo.environ as pyo
from pyomo.environ import *
import pyomo.dae as dae
import numpy as np

Example GP:

In [2]:
Time = np.linspace(0, 99, 100)
Temperature = Time ** 2
Pressure = np.sin(Time)
Energy = Temperature + Pressure

try:
    GP = FoKLRoutines.load('FoKL_model_test.fokl')
except Exception as exception:
    GP = FoKLRoutines.FoKL(kernel=1, UserWarnings=False)
    _ = GP.fit([Temperature, Pressure], Energy, clean=True)
    GP.save('FoKL_model_test.fokl')

User inputs:

In [3]:
draws = 5
gp_name = 'GP name test'

mtx = GP.mtx
betas = GP.betas
minmax = GP.minmax
phis = GP.phis

# =======================

t = [2.4, 9.1]  # acts like t_span

# - OR -

m_global = pyo.ConcreteModel('Pyomo model test')
m_global.t = dae.ContinuousSet(bounds=[2.4, 9.1])
t = m_global.t  # pass ODE time from global Pyomo model to dummy model

# ============================

m_global.T = pyo.Var(m_global.t, domain=pyo.Reals)
m_global.P = dae.DerivativeVar(m_global.T, domain=pyo.Reals)  # not true to example but demonstrates why user may need to define variables

USER_VARIABLES = [m_global.T, m_global.P]

## Create sub-model ```m``` of GP only, prior to merging with ```m_global```

- ```xvar``` and ```yvar``` defined in ```m_global```

In [4]:
m = pyo.ConcreteModel(gp_name)

# Some constants:
mtx = np.array(mtx, dtype=int)  # indices/orders of basis functions (where 1 is B1 and 0 means none)

# Some sets:
m.draws = pyo.Set(initialize = range(draws))
m.terms = pyo.Set(initialize = range(mtx.shape[0] + 1))  # terms (including beta0)
m.orders = pyo.Set(initialize = np.unique(mtx[mtx != 0]))  # orders of basis functions
m.attributes = pyo.Set(initialize = range(mtx.shape[1]))  # input variables

## Define betas variables

In [5]:
m.beta = pyo.Var(m.draws, m.terms, domain=pyo.Reals)

m.beta_avg = pyo.Var(m.terms, domain=pyo.Reals)

In [6]:
def fix_betas(m, betas):
    """Fix the already-initialized Pyomo beta variables to scalar values in 'betas', using last 'betas' draw as first Pyomo draw."""
    for draw in m.draws:
        for term in m.terms:
            m.beta[draw, term].fix(betas[-(draw + 1), term])

## Define time for ODE

In [7]:
if t is not None:  # if user passed either bounds [min, max] or ContinuousSet
    if isinstance(t, list):
        m.t = dae.ContinuousSet(bounds=t)  # t acts like t_span
    elif isinstance(t, dae.ContinuousSet):
        m.t = dae.ContinuousSet(bounds=t.bounds())  # t is ContinuousSet to be copied
    else:
        raise ValueError("Input 't' must be a list of [min, max] bounds, or a 'dae.ContinuousSet'.")
else:  # ODE not requested, so single index to avoid if-else statements in internal code
    m.t = pyo.Set(initialize=range(1))

## Define normalized input variables (attributes)

In [8]:
m.x = pyo.Var(m.t, m.attributes, domain=pyo.Reals, bounds=(0, 1))

## Define phi "basis" functions

- which are not technically "bases" because Bernoulli's are not sets of vectors in same vector space (each "basis" function for Bernoulli is single "vector" in its own vector space, whereas Cubic Splines are 500 sets of 499 4D vectors)
    - hence, using ```orders``` not ```bases```

Pyomo expression as callable function:
- https://groups.google.com/g/pyomo-forum/c/LJkkyHxZT1A/m/NBa7W9TWL-kJ

Example of syntax with ContinuousSet:
- https://pyomo.readthedocs.io/en/stable/modeling_extensions/dae.html#using-the-simulator

In [9]:
nj = []  # list of [order, attribute] combinations used in GP
for attribute in m.attributes:
    orders_j = np.unique(mtx[:, attribute])
    if any(orders_j != 0):
        for order_j in orders_j[orders_j != 0]:
            nj.append([order_j, attribute])

In [10]:
m.phi = pyo.Var(m.t, nj, domain=pyo.Reals)

def _eq_phi(m, t, n, j):
    """FoKL's 'basis' functions."""
    nm1 = n - 1  # Python indexing, since n=1 refers to B1 which is phis[0]
    return m.phi[t, n, j] == phis[nm1][0] + sum(phis[nm1][k] * m.x[t, j] ** k for k in range(1, len(phis[nm1])))

m.constr_phi = pyo.Constraint(m.t, nj, rule=_eq_phi)

## Build GP expression

Draws:

In [11]:
m.y = pyo.Var(m.t, m.draws, domain=pyo.Reals)

def _eq_y(m, t, draw):
    """FoKL's GP equation."""
    y = m.beta[draw, 0]  # initialize
    
    for term in range(1, len(m.terms)):  # == m.terms[1::]
        y_term = m.beta[draw, term]

        for j in m.attributes:
            n = mtx[term - 1, j]

            if n != 0:  # since 0 means none
                y_term *= m.phi[t, n, j]

        y += y_term

    return m.y[t, draw] == y

m.constr_y = pyo.Constraint(m.t, m.draws, rule=_eq_y)

Average:

In [12]:
m.y_avg = pyo.Var(m.t, domain=pyo.Reals)

def _eq_y_avg(m, t):
    """FoKL's GP equation, averaged across draws."""
    y = m.beta_avg[0]  # initialize
    
    for term in range(1, len(m.terms)):  # == m.terms[1::]
        y_term = m.beta_avg[term]

        for j in m.attributes:
            n = mtx[term - 1, j]

            if n != 0:  # since 0 means none
                y_term *= m.phi[t, n, j]

        y += y_term

    return m.y_avg[t] == y

m.constr_y_avg = pyo.Constraint(m.t, rule=_eq_y_avg)

Standard deviation:

In [13]:
m.y_std = pyo.Var(m.t, domain=pyo.Reals)

def _eq_y_std(m, t):
    """Standard deviation of FoKL's GP equation draws."""
    return m.y_std[t] == sqrt(sum(m.y[t, draw] ** 2 for draw in m.draws) / len(m.draws) + 1e-9)

m.constr_y_std = pyo.Constraint(m.t, rule=_eq_y_std)

---

## Apply normalization

User must define input variables (instead of ```xvars```). The following should somehow be something like:

```python
m_global = fokl_to_pyomo.normalize(m_global, USER_VARIABLES, minmax, igp)
```

In [14]:
USER_VARIABLES = [m_global.T, m_global.P]

def _eq_norm(m, t, j):
    """Normalization constraint."""
    return m.x[t, j] == (USER_VARIABLES[j][t] - minmax[j][0]) / (minmax[j][1] - minmax[j][0])

m.constr_norm = pyo.Constraint(m.t, m.attributes, rule=_eq_norm)

---

## Port dummy model into global model

- https://or.stackexchange.com/questions/7128/can-we-take-the-constraints-from-one-model-and-plug-them-into-the-other-model-in

In [15]:
m_global.GP0 = m